# Lesson 21 - MySQL with Python

**Goal:** connect to a MySQL server, model employee records, perform CRUD, use joins and
aggregations, and control transactions with MySQL Connector/Python.

**Level:** intermediate. Complete functions, validation, Decimal, and Lesson 19's SQL introduction
first. SQLite is embedded; MySQL is a separate server with databases, accounts, and permissions.
This lesson uses the classic `mysql.connector` API, not the separate X DevAPI.

Offline cells run without a driver or server. The live lab is **off by default** and clearly
reports when skipped. After setup, set `RUN_MYSQL = True` and run all cells. Live exercises use
connection-scoped temporary tables and synthetic data; closing the connection removes them.
No SQL result is claimed to be verified when the live lab is skipped.

## 1. Install the driver and prepare a server

In PowerShell, activate the project's virtual environment and install the project dependencies (including the MySQL driver):

```powershell
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -c "import mysql.connector; print(mysql.connector.__version__)"
```

Install and start MySQL Community Server separately using its official Windows installer or
your existing MySQL installation. This notebook targets MySQL 8.4; MySQL 8.0.16+ is needed for
enforced CHECK constraints. MySQL Workbench is an optional graphical client, not the server.
The pip driver alone does not install or start a server.

Connect to your local MySQL server as `root` using password `xyz123`, then create
the learning database once in the mysql client or Workbench:

```sql
CREATE DATABASE IF NOT EXISTS python_practice
  CHARACTER SET utf8mb4;
```

The notebook uses the existing root account; it does not change the server's account password.
Your local server must already accept these credentials.

Official references: [server installation](https://dev.mysql.com/doc/refman/8.4/en/windows-installation.html),
[driver installation](https://dev.mysql.com/doc/connector-python/en/connector-python-installation-binary.html).

## 2. Local connection settings

The code below uses user `root` and password `xyz123` for this local learning setup.
The default host is `localhost`, port is `3306`, and database is `python_practice`.
No password prompt or MYSQL_USER setting is required. Host, port, and database can still
be overridden through environment variables set before starting Jupyter.

Install the driver and start the server, then set `RUN_MYSQL = True` to execute the live lab.
The notebook configures a client connection; it does not create or reset the root account.


In [ ]:
import os
from decimal import Decimal

RUN_MYSQL = False  # Set True after installing the driver and starting the local server.

def connection_settings(environ):
    port = int(environ.get("MYSQL_PORT", "3306"))
    if not 1 <= port <= 65535:
        raise ValueError("MYSQL_PORT must be between 1 and 65535")
    database = environ.get("MYSQL_DATABASE", "python_practice")
    if not database.strip():
        raise ValueError("MYSQL_DATABASE cannot be blank")
    return {"host": environ.get("MYSQL_HOST", "localhost"),
            "port": port, "database": database,
            "user": "root", "password": "dpl123", "autocommit": False,
            "connection_timeout": 10, "charset": "utf8mb4"}

sample_settings = connection_settings({})
assert sample_settings["port"] == 3306
assert sample_settings["autocommit"] is False
assert sample_settings["user"] == "root"
assert sample_settings["password"] == "dpl123"
print("Local connection settings validated offline; no connection attempted.")


Local connection settings validated offline; no connection attempted.


## 3. Schema and money types

Primary keys uniquely identify records; foreign keys relate employees to departments.
Use DECIMAL for exact fixed-point salary amounts rather than FLOAT. Connector/Python converts
DECIMAL results to Python Decimal. Validate in Python and enforce constraints in SQL too.

The live lab creates a temporary employee table in InnoDB. Temporary tables are visible only
to their connection and disappear when it closes. MySQL temporary InnoDB tables do not support
foreign keys, so the join lesson uses permanent-table DDL shown separately below. A CHECK
constraint enforces nonnegative basic pay. DDL has distinct transaction rules: do not assume
ROLLBACK undoes table creation. In particular, temporary-table creation is not rolled back.

In [2]:
CREATE_EMPLOYEES = """
CREATE TEMPORARY TABLE lesson21_employees (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(100) NOT NULL,
    department VARCHAR(50) NOT NULL,
    basic_pay DECIMAL(12, 2) NOT NULL,
    CONSTRAINT lesson21_basic_nonnegative CHECK (basic_pay >= 0)
) ENGINE=InnoDB
"""
INSERT_EMPLOYEE = """INSERT INTO lesson21_employees
    (employee_id, employee_name, department, basic_pay)
    VALUES (%s, %s, %s, %s)"""
SEED_ROWS = [
    (101, "Asha", "Finance", Decimal("52500.00")),
    (102, "Arun", "IT", Decimal("75000.00")),
    (103, "Neha", "Finance", Decimal("60000.00")),
]
assert sum(row[3] for row in SEED_ROWS) == Decimal("187500.00")
print("Prepared schema and three sample records; SQL has not been executed.")

Prepared schema and three sample records; SQL has not been executed.


## 4. Parameterized CRUD functions

Connector/Python uses `%s` value placeholders for every supported value type, not SQLite's `?`.
Pass a tuple or mapping separately; do not use f-strings, Python `%` interpolation, or quote the
placeholder. For a single parameter use `(employee_id,)`. Parameters cannot substitute SQL
identifiers such as a table name: keep those fixed or choose them from an explicit allowlist.

The functions below do not commit. The caller controls the transaction. A dictionary cursor
makes column names explicit. Consume query results before issuing another query on that cursor.
Reference: [execute parameters](https://dev.mysql.com/doc/connector-python/en/connector-python-api-mysqlcursor-execute.html).

In [3]:
def insert_employee(cursor, row):
    cursor.execute(INSERT_EMPLOYEE, row)

def get_employee(cursor, employee_id):
    cursor.execute("SELECT employee_id, employee_name, department, basic_pay "
                   "FROM lesson21_employees WHERE employee_id = %s", (employee_id,))
    rows = cursor.fetchall()
    return rows[0] if rows else None

def update_basic(cursor, employee_id, basic_pay):
    if not isinstance(basic_pay, Decimal) or not basic_pay.is_finite() or basic_pay < 0:
        raise ValueError("Basic pay must be a finite nonnegative Decimal")
    cursor.execute("UPDATE lesson21_employees SET basic_pay = %s WHERE employee_id = %s",
                   (basic_pay, employee_id))
    return cursor.rowcount

def delete_employee(cursor, employee_id):
    cursor.execute("DELETE FROM lesson21_employees WHERE employee_id = %s", (employee_id,))
    return cursor.rowcount

print("CRUD functions defined; connection and transaction remain the caller's responsibility.")

CRUD functions defined; connection and transaction remain the caller's responsibility.


## 5. Offline check: values are passed separately

This recording cursor checks how our Python functions call execute. It is a test double,
not a database engine: it cannot validate SQL syntax, permissions, constraints, or transactions.
The subsequent live lab is needed to test those behaviors.

In [4]:
class RecordingCursor:
    def __init__(self):
        self.calls = []
        self.rowcount = 1

    def execute(self, statement, parameters):
        self.calls.append((statement, parameters))

    def fetchall(self):
        return []

recording = RecordingCursor()
quoted_name = "O'Neil"
insert_employee(recording, (999, quoted_name, "IT", Decimal("50000")))
statement, parameters = recording.calls[-1]
assert quoted_name not in statement
assert parameters[1] == quoted_name
assert get_employee(recording, 999) is None
assert recording.calls[-1][1] == (999,)
assert update_basic(recording, 999, Decimal("51000")) == 1
assert delete_employee(recording, 999) == 1
print("Offline parameter-binding checks passed; no MySQL server was used.")

Offline parameter-binding checks passed; no MySQL server was used.


## 6. Live lab: create, read, update, delete, and rollback

Run this cell after changing RUN_MYSQL to True. It opens a fresh connection each time, creates
temporary tables, seeds the same synthetic rows, and always closes cursor and connection.
Each successful write group explicitly commits. Errors explicitly roll back and are re-raised.

The duplicate-ID example rolls back **both** attempted inserts in that transaction. Catch the
IntegrityError outside the transaction's work, then verify the first insert was also undone.
The UPDATE uses a genuinely different value: rowcount may be zero for an unchanged value and
does not always mean a missing employee. A SELECT can start a transaction too; finish it before
calling start_transaction again. Closing the connection removes this lesson's temporary table.
Reference: [rollback](https://dev.mysql.com/doc/connector-python/en/connector-python-api-mysqlconnection-rollback.html),
[temporary tables](https://dev.mysql.com/doc/refman/8.4/en/create-temporary-table.html).

In [5]:
def run_mysql_lab(config):
    import mysql.connector
    from mysql.connector import IntegrityError

    connection = mysql.connector.connect(**config)
    cursor = None
    try:
        cursor = connection.cursor(dictionary=True)
        cursor.execute(CREATE_EMPLOYEES)
        connection.commit()
        connection.start_transaction()
        cursor.executemany(INSERT_EMPLOYEE, SEED_ROWS)
        connection.commit()

        # READ and UPDATE
        original = get_employee(cursor, 101)
        assert original["basic_pay"] == Decimal("52500.00")
        assert update_basic(cursor, 101, Decimal("55000.00")) == 1
        connection.commit()
        assert get_employee(cursor, 101)["basic_pay"] == Decimal("55000.00")
        connection.commit()

        # CREATE and DELETE an extra row; seed employees remain.
        insert_employee(cursor, (104, "O'Neil", "IT", Decimal("50000.00")))
        assert get_employee(cursor, 104)["employee_name"] == "O'Neil"
        assert delete_employee(cursor, 104) == 1
        connection.commit()
        assert get_employee(cursor, 104) is None
        connection.commit()

        # Two inserts form one transaction: duplicate failure undoes the first too.
        connection.start_transaction()
        try:
            insert_employee(cursor, (999, "Rollback sample", "IT", Decimal("100.00")))
            insert_employee(cursor, (101, "Duplicate", "IT", Decimal("100.00")))
        except IntegrityError:
            connection.rollback()
        else:
            connection.rollback()
            raise AssertionError("Expected a duplicate primary-key error")
        assert get_employee(cursor, 999) is None
        connection.commit()

        # Aggregation uses the updated Asha salary (55,000).
        cursor.execute("SELECT department, COUNT(*) AS employee_count, "
                       "SUM(basic_pay) AS total_basic FROM lesson21_employees "
                       "GROUP BY department ORDER BY department")
        summary = cursor.fetchall()
        assert summary[0]["department"] == "Finance"
        assert summary[0]["employee_count"] == 2
        assert summary[0]["total_basic"] == Decimal("115000.00")
        assert summary[1]["total_basic"] == Decimal("75000.00")
        connection.commit()
        print("Live MySQL CRUD, Decimal round trip, aggregation, and rollback checks passed.")
        return summary
    except Exception:
        connection.rollback()
        raise
    finally:
        try:
            if cursor is not None:
                cursor.close()
        finally:
            connection.close()

if RUN_MYSQL:
    config = connection_settings(os.environ)
    try:
        mysql_summary = run_mysql_lab(config)
        print(mysql_summary)
    finally:
        config.pop("password", None)
else:
    print("SKIPPED live MySQL lab: enable RUN_MYSQL after driver and server setup.")

SKIPPED live MySQL lab: enable RUN_MYSQL after driver and server setup.


## 7. Joins, foreign keys, and persistent schemas

For a persistent application, normalize departments into a separate table and relate employees
using a foreign key. The following is **SQL to run deliberately in a learning schema**, not
an automatically executed notebook cell. The earlier student account needs administrator-created
tables for this extension. Use distinct names from the temporary lab.

```sql
CREATE TABLE lesson21_departments (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(50) NOT NULL UNIQUE
) ENGINE=InnoDB;
CREATE TABLE lesson21_staff (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(100) NOT NULL,
    department_id INT,
    basic_pay DECIMAL(12,2) NOT NULL CHECK (basic_pay >= 0),
    FOREIGN KEY (department_id) REFERENCES lesson21_departments(department_id)
) ENGINE=InnoDB;

SELECT s.employee_id, s.employee_name, d.department_name
FROM lesson21_staff AS s
LEFT JOIN lesson21_departments AS d
  ON s.department_id = d.department_id
ORDER BY s.employee_id;
```

INNER JOIN returns matching departments; LEFT JOIN also retains employees with a null department.
Use explicit columns and aliases, qualify ambiguous fields, and define behavior for missing links.
Aggregate at the intended level: joining one employee to several related rows can multiply totals.
An index helps relevant lookups but adds write cost; inspect EXPLAIN rather than assuming every
column needs an index. SQL alone does not calculate the course's full payroll formulas here.

## 8. Troubleshooting and SQLite differences

| Symptom | Check |
| --- | --- |
| ModuleNotFoundError | Install requirements.txt in the notebook's selected Python environment |
| Connection refused or timeout | Server running, host, port, and local connection settings |
| Access denied | User, password, matching account host, and grants |
| Unknown database | Create the learning database before connecting |
| Table permission error | Temporary-table privilege for the lab; admin-created tables for the persistent extension |
| Unread result found | Fetch the prior SELECT results before another statement |
| Changes disappear | Commit successful transactions; the lab's temporary tables disappear on close by design |

SQLite uses `?` placeholders, is embedded, and supports different types and SQL details.
MySQL uses its own server, accounts, `%s` placeholders in this driver, DECIMAL columns, and
storage-engine transaction behavior. Do not mechanically replace the SQLite connection string
and assume the rest is portable.

## Practice lab

Use only your learning database and synthetic employees. Keep credentials out of answer cells.
Exercises 1-2 can run offline; exercises 3-8 need the configured server. Define live exercises
inside a function owning a connection, and close it in finally. Use fresh temporary tables where
possible. Persistent joins require the separate administrator-created schema above.

### Exercise 1. Connection validation

Test an empty settings dictionary, a blank MYSQL_DATABASE, a nonnumeric port, and ports 0 and 65536 without opening a connection.

**Check:** Empty settings use the local defaults and root account; a blank database and invalid ports raise ValueError.

In [6]:
# Write your solution here. Live exercises require your local MySQL setup.

### Exercise 2. Parameter contract

Use RecordingCursor to verify an apostrophe-containing name is passed in the parameter tuple and never inserted into SQL text.

**Check:** The statement remains unchanged; the name is preserved as data.

In [7]:
# Write your solution here. Live exercises require your local MySQL setup.

### Exercise 3. Employee CRUD

Create a fresh temporary table, insert ID 201, read it, update basic from 60,000 to 62,000, and delete it. Commit each intended write group.

**Check:** Read-after-update returns Decimal("62000.00"); read-after-delete returns None.

In [8]:
# Write your solution here. Live exercises require your local MySQL setup.

### Exercise 4. Atomic batch

In one transaction insert a new ID and then a duplicate ID. Roll back on failure.

**Check:** The new ID is also absent afterward; previously committed seed data remains.

In [9]:
# Write your solution here. Live exercises require your local MySQL setup.

### Exercise 5. Department summary

Group the original three seed rows by department before any salary update.

**Check:** Finance count 2, total 112,500.00; IT count 1, total 75,000.00. The worked live lab instead expects Finance 115,000.00 after its update.

In [10]:
# Write your solution here. Live exercises require your local MySQL setup.

### Exercise 6. Foreign-key joins

In the persistent extension, create Finance and IT departments and three employees, one with null department. Compare inner and left joins.

**Check:** Inner join omits the unassigned employee; left join retains it with a null department name.

In [11]:
# Write your solution here. Live exercises require your local MySQL setup.

### Exercise 7. Constraints and money

Test duplicate IDs, negative basic pay, a missing name, and a Decimal amount with cents.

**Check:** Invalid rows fail and roll back; an amount such as 52500.25 round-trips as Decimal without float conversion.

In [12]:
# Write your solution here. Live exercises require your local MySQL setup.

### Exercise 8. Payroll integration

Read employee basic pay as Decimal, calculate salary with Lesson 20 rules, and export JSON with monetary strings.

**Check:** Basic 52,500 plus electricity 1,000 gives PF 9,324.00, monthly tax 0.00, and monthly net 77,776.00.

In [13]:
# Write your solution here. Live exercises require your local MySQL setup.

## Chapter project: a MySQL-backed payroll repository

Create an EmployeeRepository whose methods use parameterized statements. Keep transaction control
at the workflow boundary rather than committing inside every repository method. Read valid
employee records, calculate results with the Decimal payroll function, and export a JSON report.
Use composition to inject the repository into the report workflow; a report is not an employee.

Verify missing IDs, apostrophes in names, duplicate IDs, constraint failures, an empty result set,
and rollback of a partially processed batch. For persistent tables, reconnect and verify committed
data survives, while rolled-back data is absent. Document the schema, grants, rounding policy,
and which tests require a live server. Do not claim offline cursor tests verify MySQL transactions.

In [14]:
# Build your repository, report workflow, and live integration checks here.

## Reflection and references

1. Why does installing the driver not install a database server?
2. What do commit, rollback, and closing a connection each do?
3. Why do placeholders protect values but not dynamic table names?
4. When do you need a LEFT JOIN rather than an INNER JOIN?
5. Which tests can run offline, and which must connect to MySQL?

Official sources used for the lesson:
- [Connector/Python guide](https://dev.mysql.com/doc/connector-python/en/)
- [Parameterized execute](https://dev.mysql.com/doc/connector-python/en/connector-python-api-mysqlcursor-execute.html)
- [Rollback and autocommit](https://dev.mysql.com/doc/connector-python/en/connector-python-api-mysqlconnection-rollback.html)
- [Temporary tables](https://dev.mysql.com/doc/refman/8.4/en/create-temporary-table.html)

After completion, revisit the capstone with a persistent MySQL repository and a tested OOP design.